# Mammography — post hoc alternative architectures

Exploratory evaluation of alternative U-shaped architectures under the same mammography ROI protocol.


## Étape 1 — Configuration générale


In [ ]:

from pathlib import Path
import os, json, math, random, time, hashlib, zipfile, shutil, warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

try:
    import cv2
except Exception as e:
    raise ImportError("opencv-python is required. Kaggle usually provides cv2.") from e

try:
    from scipy import ndimage as ndi
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False
    print("WARNING: scipy not available. HD95/ASD will be set to NaN.")

RUN_PRECHECK = True
RUN_VISUALIZATION = True
RUN_TRAINING = True  # Set False for a dry run or pre-check only.
QUICK_DEBUG = False  # Set True only for a short subset run; do not use this mode for final results.

RUN_UNETPP = True
RUN_CONNECTED_UNETS = True
RUN_RESIDUAL_UNET = False
RUN_CONNECTED_CLAHE_ARM = False

SEEDS = [42, 123, 2025]
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 120
PATIENCE = 15
WARMUP_EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
USE_AMP = True
USE_POS_WEIGHT = True
SAVE_PROBABILITIES = False  # Keep False to avoid excessive output size.

BASELINE = {
    "model": "swin_tiny_unet_scratch_baseline",
    "cbis_test_dice": 0.888,
    "cbis_test_iou": 0.809,
    "cbis_test_hd95": 13.98,
    "inbreast_dice": 0.849,
    "inbreast_iou": 0.755,
    "inbreast_hd95": 17.93,
}

OUT_DIR = Path("/kaggle/working/ROI256_WAVE3_ARCHITECTURES_RESULTS")
OUT_DIR.mkdir(parents=True, exist_ok=True)
for sub in ["checkpoints", "figures", "logs", "metrics", "probabilities"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

assert OUT_DIR.name == "ROI256_WAVE3_ARCHITECTURES_RESULTS", "Wrong OUT_DIR. This notebook must write to the Wave 3 folder only."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)


## Étape 2 — Détection du dataset et audit des splits


In [ ]:

def find_manifest() -> Path:
    candidates = list(Path("/kaggle/input").rglob("roi_crop_manifest.csv"))
    if not candidates:
        raise FileNotFoundError(
            "roi_crop_manifest.csv introuvable dans /kaggle/input. "
            "Ajoute le dataset ROI_Crops_256_v1 comme input Kaggle."
        )
    candidates = sorted(candidates, key=lambda p: ("ROI_Crops_256_v1" not in str(p), len(str(p))))
    return candidates[0]

MANIFEST_PATH = find_manifest()
DATA_ROOT = MANIFEST_PATH.parent
print("MANIFEST_PATH:", MANIFEST_PATH)
print("DATA_ROOT:", DATA_ROOT)

df = pd.read_csv(MANIFEST_PATH)
print("Manifest shape:", df.shape)
display(df.head())

required_cols = ["dataset", "split", "patient_id", "npz_path"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required manifest columns: {missing}")

counts = df.groupby(["dataset", "split"]).size().reset_index(name="n")
counts["n"] = counts["n"].astype(int)
display(counts)

cbis = df[df["dataset"].astype(str).str.contains("CBIS", case=False, na=False)].copy()
cbis["patient_id"] = cbis["patient_id"].astype(str)
patient_sets = {}
for s in ["train", "validation", "test"]:
    patient_sets[s] = set(cbis.loc[cbis["split"].astype(str) == s, "patient_id"].dropna())

leaks = {
    "train_vs_validation": len(patient_sets["train"] & patient_sets["validation"]),
    "train_vs_test": len(patient_sets["train"] & patient_sets["test"]),
    "validation_vs_test": len(patient_sets["validation"] & patient_sets["test"]),
}
print("CBIS patient leakage:", leaks)
if any(v > 0 for v in leaks.values()):
    raise RuntimeError(f"Patient leakage detected: {leaks}")
print("OK: no CBIS patient leakage.")

split_audit = {
    "manifest_path": str(MANIFEST_PATH),
    "data_root": str(DATA_ROOT),
    "n_rows": int(len(df)),
    "counts_by_dataset_split": counts.to_dict(orient="records"),
    "patient_leakage_cbisdDSM": {str(k): int(v) for k, v in leaks.items()},
}
with open(OUT_DIR / "logs" / "split_leakage_audit.json", "w", encoding="utf-8") as f:
    json.dump(split_audit, f, indent=2, ensure_ascii=False)
counts.to_csv(OUT_DIR / "logs" / "split_counts.csv", index=False)
print("Saved split audit.")


## Étape 3 — Utilitaires : seeds, chemins NPZ, overlay et augmentation légère


In [ ]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def resolve_npz_path(npz_path: str) -> Path:
    p = Path(str(npz_path))
    candidates = [p, DATA_ROOT / p, DATA_ROOT / p.name]
    for c in candidates:
        if c.exists():
            return c
    matches = list(DATA_ROOT.rglob(p.name))
    if not matches:
        matches = list(Path("/kaggle/input").rglob(p.name))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"NPZ not found: {npz_path}")


def overlay_mask(img: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    img_u8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    rgb = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2RGB)
    ov = rgb.copy()
    ov[mask > 0] = [255, 0, 0]
    return cv2.addWeighted(ov, alpha, rgb, 1 - alpha, 0)


def apply_clahe_uint8(img_u8: np.ndarray, clip_limit=2.0, tile_grid_size=(8,8)) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(img_u8)


def light_augment(img: np.ndarray, mask: np.ndarray, rng: np.random.RandomState):
    """img float [0,1], mask uint8 {0,1}; return synchronized light augmentation."""
    h, w = img.shape
    img_aug = img.copy()
    mask_aug = mask.copy()

    if rng.rand() < 0.5:
        img_aug = np.ascontiguousarray(np.fliplr(img_aug))
        mask_aug = np.ascontiguousarray(np.fliplr(mask_aug))
    if rng.rand() < 0.1:
        img_aug = np.ascontiguousarray(np.flipud(img_aug))
        mask_aug = np.ascontiguousarray(np.flipud(mask_aug))

    if rng.rand() < 0.7:
        angle = rng.uniform(-10, 10)
        scale = rng.uniform(0.95, 1.05)
        tx = rng.uniform(-0.03, 0.03) * w
        ty = rng.uniform(-0.03, 0.03) * h
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, scale)
        M[0, 2] += tx
        M[1, 2] += ty
        img_aug = cv2.warpAffine(
            img_aug, M, (w, h), flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0
        )
        mask_aug = cv2.warpAffine(
            mask_aug, M, (w, h), flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0
        )

    return np.clip(img_aug, 0, 1).astype(np.float32), (mask_aug > 0).astype(np.uint8)


## Étape 4 — Dataset et DataLoaders


In [ ]:

class ROICropDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, train: bool = False, seed: int = 0, use_clahe: bool = False):
        self.frame = frame.reset_index(drop=True).copy()
        self.train = train
        self.seed = seed
        self.use_clahe = use_clahe
        self.paths = [resolve_npz_path(p) for p in self.frame["npz_path"].tolist()]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        z = np.load(self.paths[idx])
        img = z["image"].astype(np.float32)
        mask = z["mask"].astype(np.uint8)
        if img.ndim == 3:
            img = np.squeeze(img)
        if mask.ndim == 3:
            mask = np.squeeze(mask)
        img = np.clip(img, 0, 1)
        mask = (mask > 0).astype(np.uint8)

        if self.use_clahe:
            img_u8 = (img * 255).astype(np.uint8)
            img = apply_clahe_uint8(img_u8).astype(np.float32) / 255.0

        if self.train:
            rng = np.random.RandomState(self.seed + idx + int(time.time() // 3600) * 0)
            img, mask = light_augment(img, mask, rng)

        x = torch.from_numpy(img).float().unsqueeze(0)
        y = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0)
        return {
            "image": x,
            "mask": y,
            "sample_id": str(self.frame.iloc[idx].get("sample_id", idx)),
            "path": str(self.paths[idx]),
        }


def get_split_frames(max_debug=None):
    train_df = df[(df["dataset"].astype(str).str.contains("CBIS", case=False, na=False)) & (df["split"].astype(str) == "train")].copy()
    val_df = df[(df["dataset"].astype(str).str.contains("CBIS", case=False, na=False)) & (df["split"].astype(str) == "validation")].copy()
    test_df = df[(df["dataset"].astype(str).str.contains("CBIS", case=False, na=False)) & (df["split"].astype(str) == "test")].copy()
    ext_df = df[df["split"].astype(str).str.contains("external", case=False, na=False)].copy()
    if max_debug is not None:
        train_df = train_df.sample(min(max_debug, len(train_df)), random_state=42)
        val_df = val_df.sample(min(max_debug, len(val_df)), random_state=42)
        test_df = test_df.sample(min(max_debug, len(test_df)), random_state=42)
        ext_df = ext_df.sample(min(max_debug, len(ext_df)), random_state=42)
    return train_df, val_df, test_df, ext_df


def make_loaders(seed: int, use_clahe: bool = False):
    max_debug = 24 if QUICK_DEBUG else None
    train_df, val_df, test_df, ext_df = get_split_frames(max_debug=max_debug)
    train_ds = ROICropDataset(train_df, train=True, seed=seed, use_clahe=use_clahe)
    val_ds = ROICropDataset(val_df, train=False, seed=seed, use_clahe=use_clahe)
    test_ds = ROICropDataset(test_df, train=False, seed=seed, use_clahe=use_clahe)
    ext_ds = ROICropDataset(ext_df, train=False, seed=seed, use_clahe=use_clahe)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    ext_loader = DataLoader(ext_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader, ext_loader

train_df0, val_df0, test_df0, ext_df0 = get_split_frames()
print(len(train_df0), len(val_df0), len(test_df0), len(ext_df0))


## Étape 5 — Visualisation rapide image/masque et augmentation légère


In [ ]:

if RUN_VISUALIZATION:
    sample_rows = train_df0.sample(n=min(4, len(train_df0)), random_state=42).reset_index(drop=True)
    n_aug = 3
    fig, axes = plt.subplots(len(sample_rows), 1+n_aug, figsize=(4*(1+n_aug), 4*len(sample_rows)))
    if len(sample_rows) == 1:
        axes = np.expand_dims(axes, 0)
    for r in range(len(sample_rows)):
        p = resolve_npz_path(sample_rows.iloc[r]["npz_path"])
        z = np.load(p)
        img = np.squeeze(z["image"]).astype(np.float32)
        mask = (np.squeeze(z["mask"]) > 0).astype(np.uint8)
        img = np.clip(img, 0, 1)
        axes[r,0].imshow(overlay_mask(img, mask))
        axes[r,0].set_title(f"Original\n{p.name}", fontsize=8)
        axes[r,0].axis("off")
        for c in range(1, 1+n_aug):
            rng = np.random.RandomState(1000 + r*10 + c)
            aimg, amask = light_augment(img, mask, rng)
            axes[r,c].imshow(overlay_mask(aimg, amask))
            axes[r,c].set_title(f"Light aug {c}")
            axes[r,c].axis("off")
    plt.tight_layout()
    outp = OUT_DIR / "figures" / "01_wave3_light_augmentation_check.png"
    plt.savefig(outp, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", outp)


## Étape 6 — Définition des modèles : UNet++, Connected-UNets, Residual U-Net optionnel


In [ ]:

def group_norm(channels: int, max_groups: int = 8):
    g = min(max_groups, channels)
    while channels % g != 0 and g > 1:
        g -= 1
    return nn.GroupNorm(g, channels)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, residual=False):
        super().__init__()
        self.residual = residual and (in_ch == out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            group_norm(out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            group_norm(out_ch),
        )
        self.act = nn.SiLU(inplace=True)
        self.proj = None if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)
    def forward(self, x):
        y = self.net(x)
        if self.residual:
            y = y + x
        return self.act(y)

class UNetPP(nn.Module):
    """UNet++ with optional deep supervision. Final output is x0_4."""
    def __init__(self, in_ch=1, out_ch=1, base=32, deep_supervision=True):
        super().__init__()
        nb = [base, base*2, base*4, base*8, base*16]
        self.deep_supervision = deep_supervision
        self.pool = nn.MaxPool2d(2, 2)
        self.up = lambda x: F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)

        self.conv0_0 = ConvBlock(in_ch, nb[0])
        self.conv1_0 = ConvBlock(nb[0], nb[1])
        self.conv2_0 = ConvBlock(nb[1], nb[2])
        self.conv3_0 = ConvBlock(nb[2], nb[3])
        self.conv4_0 = ConvBlock(nb[3], nb[4])

        self.conv0_1 = ConvBlock(nb[0] + nb[1], nb[0])
        self.conv1_1 = ConvBlock(nb[1] + nb[2], nb[1])
        self.conv2_1 = ConvBlock(nb[2] + nb[3], nb[2])
        self.conv3_1 = ConvBlock(nb[3] + nb[4], nb[3])

        self.conv0_2 = ConvBlock(nb[0]*2 + nb[1], nb[0])
        self.conv1_2 = ConvBlock(nb[1]*2 + nb[2], nb[1])
        self.conv2_2 = ConvBlock(nb[2]*2 + nb[3], nb[2])

        self.conv0_3 = ConvBlock(nb[0]*3 + nb[1], nb[0])
        self.conv1_3 = ConvBlock(nb[1]*3 + nb[2], nb[1])

        self.conv0_4 = ConvBlock(nb[0]*4 + nb[1], nb[0])

        self.final1 = nn.Conv2d(nb[0], out_ch, 1)
        self.final2 = nn.Conv2d(nb[0], out_ch, 1)
        self.final3 = nn.Conv2d(nb[0], out_ch, 1)
        self.final4 = nn.Conv2d(nb[0], out_ch, 1)

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x2_0 = self.conv2_0(self.pool(x1_0))
        x3_0 = self.conv3_0(self.pool(x2_0))
        x4_0 = self.conv4_0(self.pool(x3_0))

        x0_1 = self.conv0_1(torch.cat([x0_0, self.up(x1_0)], 1))
        x1_1 = self.conv1_1(torch.cat([x1_0, self.up(x2_0)], 1))
        x2_1 = self.conv2_1(torch.cat([x2_0, self.up(x3_0)], 1))
        x3_1 = self.conv3_1(torch.cat([x3_0, self.up(x4_0)], 1))

        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.up(x1_1)], 1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.up(x2_1)], 1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.up(x3_1)], 1))

        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.up(x1_2)], 1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.up(x2_2)], 1))

        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.up(x1_3)], 1))

        if self.training and self.deep_supervision:
            return [self.final1(x0_1), self.final2(x0_2), self.final3(x0_3), self.final4(x0_4)]
        return self.final4(x0_4)

class UNetSmall(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32, residual=False):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base, residual=False)
        self.e2 = ConvBlock(base, base*2, residual=residual)
        self.e3 = ConvBlock(base*2, base*4, residual=residual)
        self.e4 = ConvBlock(base*4, base*8, residual=residual)
        self.center = ConvBlock(base*8, base*16, residual=residual)
        self.pool = nn.MaxPool2d(2)
        self.d4 = ConvBlock(base*16 + base*8, base*8, residual=residual)
        self.d3 = ConvBlock(base*8 + base*4, base*4, residual=residual)
        self.d2 = ConvBlock(base*4 + base*2, base*2, residual=residual)
        self.d1 = ConvBlock(base*2 + base, base, residual=residual)
        self.out = nn.Conv2d(base, out_ch, 1)
    def up(self, x):
        return F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        c = self.center(self.pool(e4))
        d4 = self.d4(torch.cat([self.up(c), e4], 1))
        d3 = self.d3(torch.cat([self.up(d4), e3], 1))
        d2 = self.d2(torch.cat([self.up(d3), e2], 1))
        d1 = self.d1(torch.cat([self.up(d2), e1], 1))
        return self.out(d1)

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, dilations=(1, 3, 5)):
        super().__init__()
        branches = []
        for d in dilations:
            branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=d, dilation=d, bias=False),
                group_norm(out_ch),
                nn.SiLU(inplace=True)
            ))
        self.branches = nn.ModuleList(branches)
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * len(dilations), out_ch, 1, bias=False),
            group_norm(out_ch),
            nn.SiLU(inplace=True)
        )
    def forward(self, x):
        return self.project(torch.cat([b(x) for b in self.branches], 1))

class ConnectedUNets(nn.Module):
    """Connected-UNets inspired by Connected-SegNets: first U-Net predicts, second U-Net refines via ASPP bridge."""
    def __init__(self, in_ch=1, out_ch=1, base=24, deep_supervision=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.stage1 = UNetSmall(in_ch=in_ch, out_ch=out_ch, base=base, residual=False)
        self.bridge = ASPP(in_ch + out_ch, base, dilations=(1, 3, 5))
        self.stage2 = UNetSmall(in_ch=base, out_ch=out_ch, base=base, residual=False)
    def forward(self, x):
        logits1 = self.stage1(x)
        p1 = torch.sigmoid(logits1)
        z = self.bridge(torch.cat([x, p1], 1))
        logits2 = self.stage2(z)
        if self.training and self.deep_supervision:
            return [logits1, logits2]
        return logits2

class ResidualUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.net = UNetSmall(in_ch=in_ch, out_ch=out_ch, base=base, residual=True)
    def forward(self, x):
        return self.net(x)


def build_model(model_name: str):
    if model_name == "unetpp":
        return UNetPP(in_ch=1, out_ch=1, base=32, deep_supervision=True)
    if model_name == "connected_unets":
        return ConnectedUNets(in_ch=1, out_ch=1, base=24, deep_supervision=True)
    if model_name == "residual_unet":
        return ResidualUNet(in_ch=1, out_ch=1, base=32)
    if model_name == "connected_unets_clahe":
        return ConnectedUNets(in_ch=1, out_ch=1, base=24, deep_supervision=True)
    raise ValueError(model_name)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

models_to_precheck = []
if RUN_UNETPP: models_to_precheck.append("unetpp")
if RUN_CONNECTED_UNETS: models_to_precheck.append("connected_unets")
if RUN_RESIDUAL_UNET: models_to_precheck.append("residual_unet")
if RUN_CONNECTED_CLAHE_ARM: models_to_precheck.append("connected_unets_clahe")

param_rows = []
for name in models_to_precheck:
    m = build_model(name).to(DEVICE)
    n_params = count_parameters(m)
    with torch.no_grad():
        y = m(torch.randn(1,1,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE))
        if isinstance(y, list): y_shape = list(y[-1].shape)
        else: y_shape = list(y.shape)
    param_rows.append({"model": name, "n_params": int(n_params), "output_shape": str(y_shape)})
    del m
    torch.cuda.empty_cache()

param_df = pd.DataFrame(param_rows)
display(param_df)
param_df.to_csv(OUT_DIR / "logs" / "model_parameter_counts.csv", index=False)


## Étape 7 — Pertes et métriques


In [ ]:

def compute_pos_weight_from_loader(loader, max_batches=None):
    pos = 0.0
    neg = 0.0
    for bi, batch in enumerate(loader):
        y = batch["mask"]
        pos += float(y.sum())
        neg += float(y.numel() - y.sum())
        if max_batches is not None and bi + 1 >= max_batches:
            break
    pw = neg / max(pos, 1.0)
    return float(np.clip(pw, 1.0, 20.0))

class CompositeLoss(nn.Module):
    def __init__(self, pos_weight: Optional[float] = None):
        super().__init__()
        if pos_weight is not None:
            self.register_buffer("pos_weight", torch.tensor([pos_weight], dtype=torch.float32))
        else:
            self.pos_weight = None

    def dice_loss(self, logits, target, eps=1e-6):
        p = torch.sigmoid(logits)
        dims = (1,2,3)
        inter = (p * target).sum(dims)
        den = p.sum(dims) + target.sum(dims)
        dice = (2*inter + eps) / (den + eps)
        return 1 - dice.mean()

    def focal_tversky_loss(self, logits, target, alpha=0.3, beta=0.7, gamma=0.75, eps=1e-6):
        p = torch.sigmoid(logits)
        dims = (1,2,3)
        tp = (p * target).sum(dims)
        fp = (p * (1-target)).sum(dims)
        fn = ((1-p) * target).sum(dims)
        tversky = (tp + eps) / (tp + alpha*fp + beta*fn + eps)
        return torch.pow((1 - tversky), gamma).mean()

    def single_loss(self, logits, target):
        pos_w = self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_w)
        dl = self.dice_loss(logits, target)
        ft = self.focal_tversky_loss(logits, target)
        return bce + dl + ft

    def forward(self, outputs, target):
        if isinstance(outputs, (list, tuple)):
            if len(outputs) == 4:
                weights = [0.2, 0.3, 0.5, 1.0]
            elif len(outputs) == 2:
                weights = [0.4, 1.0]
            else:
                weights = [1.0] * len(outputs)
            denom = sum(weights)
            return sum(w * self.single_loss(o, target) for w, o in zip(weights, outputs)) / denom
        return self.single_loss(outputs, target)


def final_logits(outputs):
    if isinstance(outputs, (list, tuple)):
        return outputs[-1]
    return outputs


def binary_metrics_np(pred: np.ndarray, gt: np.ndarray, eps=1e-7):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    dice = (2*tp + eps) / (2*tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    empty_pred = int(pred.sum() == 0)
    return dice, iou, precision, recall, empty_pred


def surface_distances(mask1: np.ndarray, mask2: np.ndarray):
    if not SCIPY_OK:
        return np.array([np.nan], dtype=np.float32)
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)
    if mask1.sum() == 0 and mask2.sum() == 0:
        return np.array([0.0], dtype=np.float32)
    if mask1.sum() == 0 or mask2.sum() == 0:
        diag = math.sqrt(mask1.shape[0]**2 + mask1.shape[1]**2)
        return np.array([diag], dtype=np.float32)
    struct = ndi.generate_binary_structure(2, 1)
    s1 = np.logical_xor(mask1, ndi.binary_erosion(mask1, structure=struct, border_value=0))
    s2 = np.logical_xor(mask2, ndi.binary_erosion(mask2, structure=struct, border_value=0))
    if s1.sum() == 0 or s2.sum() == 0:
        return np.array([0.0], dtype=np.float32)
    dt2 = ndi.distance_transform_edt(~s2)
    dt1 = ndi.distance_transform_edt(~s1)
    d12 = dt2[s1]
    d21 = dt1[s2]
    return np.concatenate([d12, d21]).astype(np.float32)


def hd95_asd(pred: np.ndarray, gt: np.ndarray):
    d = surface_distances(pred, gt)
    return float(np.percentile(d, 95)), float(np.mean(d))


def aggregate_metrics(probs: np.ndarray, masks: np.ndarray, threshold: float) -> Dict[str, float]:
    preds = probs >= threshold
    rows = []
    for i in range(len(masks)):
        dice, iou, prec, rec, empty = binary_metrics_np(preds[i], masks[i])
        hd95, asd = hd95_asd(preds[i], masks[i])
        rows.append((dice, iou, prec, rec, hd95, asd, empty))
    arr = np.array(rows, dtype=np.float64)
    return {
        "dice": float(np.nanmean(arr[:,0])),
        "iou": float(np.nanmean(arr[:,1])),
        "precision": float(np.nanmean(arr[:,2])),
        "recall": float(np.nanmean(arr[:,3])),
        "hd95": float(np.nanmean(arr[:,4])),
        "asd": float(np.nanmean(arr[:,5])),
        "empty_pred_rate": float(np.nanmean(arr[:,6])),
    }


def select_threshold_on_validation(probs: np.ndarray, masks: np.ndarray):
    thresholds = np.round(np.arange(0.05, 0.951, 0.05), 2)
    best_t, best_d = None, -1
    sweep = []
    for t in thresholds:
        met = aggregate_metrics(probs, masks, float(t))
        sweep.append({"threshold": float(t), "val_dice": met["dice"], "val_iou": met["iou"]})
        if met["dice"] > best_d:
            best_d = met["dice"]
            best_t = float(t)
    return best_t, pd.DataFrame(sweep)


## Étape 8 — Boucle d'entraînement et prédiction


In [ ]:

def make_optimizer(model):
    return torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def make_scheduler(optimizer, max_epochs):
    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return float(epoch + 1) / float(max(1, WARMUP_EPOCHS))
        progress = float(epoch - WARMUP_EPOCHS) / float(max(1, max_epochs - WARMUP_EPOCHS))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

@torch.no_grad()
def evaluate_loss(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)
        y = batch["mask"].to(DEVICE, non_blocking=True)
        out = model(x)
        loss = criterion(out, y)
        total += float(loss.item()) * x.size(0)
        n += x.size(0)
    return total / max(n, 1)

@torch.no_grad()
def predict_probs_and_masks(model, loader):
    model.eval()
    all_probs = []
    all_masks = []
    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)
        y = batch["mask"].cpu().numpy()[:,0].astype(np.uint8)
        out = model(x)
        logits = final_logits(out)
        probs = torch.sigmoid(logits).detach().cpu().numpy()[:,0].astype(np.float32)
        all_probs.append(probs)
        all_masks.append(y)
    return np.concatenate(all_probs, axis=0), np.concatenate(all_masks, axis=0)


def train_one_run(model_name: str, seed: int, use_clahe: bool = False):
    set_seed(seed)
    max_epochs = 3 if QUICK_DEBUG else MAX_EPOCHS
    patience = 2 if QUICK_DEBUG else PATIENCE
    print(f"\n=== Training {model_name} seed={seed} use_clahe={use_clahe} epochs={max_epochs} ===")

    train_loader, val_loader, test_loader, ext_loader = make_loaders(seed, use_clahe=use_clahe)
    pos_weight = compute_pos_weight_from_loader(train_loader) if USE_POS_WEIGHT else None
    print("pos_weight:", pos_weight)
    criterion = CompositeLoss(pos_weight=pos_weight).to(DEVICE)
    model = build_model(model_name).to(DEVICE)
    n_params = count_parameters(model)
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, max_epochs)
    scaler = GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

    best_val_loss = float("inf")
    best_epoch = -1
    epochs_no_improve = 0
    train_log = []
    ckpt_best = OUT_DIR / "checkpoints" / f"{model_name}_seed{seed}_best.pt"

    for epoch in range(max_epochs):
        model.train()
        total_loss, n_seen = 0.0, 0
        t0 = time.time()
        for batch in train_loader:
            x = batch["image"].to(DEVICE, non_blocking=True)
            y = batch["mask"].to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(USE_AMP and DEVICE == "cuda")):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            if GRAD_CLIP is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item()) * x.size(0)
            n_seen += x.size(0)
        scheduler.step()
        train_loss = total_loss / max(n_seen, 1)
        val_loss = evaluate_loss(model, val_loader, criterion)
        lr_now = optimizer.param_groups[0]["lr"]
        elapsed = time.time() - t0

        train_log.append({
            "model": model_name, "seed": seed, "epoch": epoch+1,
            "train_loss": train_loss, "val_loss": val_loss, "lr": lr_now, "seconds": elapsed,
            "use_clahe": bool(use_clahe), "n_params": int(n_params),
        })
        print(f"epoch {epoch+1:03d}/{max_epochs} train={train_loss:.4f} val={val_loss:.4f} lr={lr_now:.2e} time={elapsed:.1f}s")

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            epochs_no_improve = 0
            torch.save({
                "model_name": model_name,
                "seed": seed,
                "use_clahe": bool(use_clahe),
                "n_params": int(n_params),
                "best_epoch": int(best_epoch),
                "best_val_loss": float(best_val_loss),
                "state_dict": model.state_dict(),
                "config": {
                    "lr": LR, "weight_decay": WEIGHT_DECAY, "max_epochs": max_epochs,
                    "patience": patience, "loss": "BCE+Dice+FocalTversky",
                    "input_channels": 1,
                }
            }, ckpt_best)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}; best_epoch={best_epoch}")
            break

    ckpt = torch.load(ckpt_best, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model.eval()

    val_probs, val_masks = predict_probs_and_masks(model, val_loader)
    test_probs, test_masks = predict_probs_and_masks(model, test_loader)
    ext_probs, ext_masks = predict_probs_and_masks(model, ext_loader)

    best_t, sweep_df = select_threshold_on_validation(val_probs, val_masks)
    sweep_path = OUT_DIR / "metrics" / f"{model_name}_seed{seed}_threshold_sweep.csv"
    sweep_df.to_csv(sweep_path, index=False)

    if SAVE_PROBABILITIES:
        np.save(OUT_DIR / "probabilities" / f"{model_name}_seed{seed}_validation_probs.npy", val_probs)
        np.save(OUT_DIR / "probabilities" / f"{model_name}_seed{seed}_test_probs.npy", test_probs)
        np.save(OUT_DIR / "probabilities" / f"{model_name}_seed{seed}_external_inbreast_probs.npy", ext_probs)

    rows = []
    for split_name, probs, masks in [
        ("validation", val_probs, val_masks),
        ("test", test_probs, test_masks),
        ("external_inbreast", ext_probs, ext_masks),
    ]:
        met = aggregate_metrics(probs, masks, best_t)
        row = {
            "model": model_name,
            "seed": seed,
            "split": split_name,
            "selected_threshold": float(best_t),
            "best_epoch": int(best_epoch),
            "best_val_loss": float(best_val_loss),
            "use_clahe": bool(use_clahe),
            "n_params": int(n_params),
            **met
        }
        rows.append(row)

    ckpt_final = OUT_DIR / "checkpoints" / f"{model_name}_seed{seed}_final_eval.pt"
    shutil.copy2(ckpt_best, ckpt_final)
    sha = sha256_file(ckpt_final)
    for row in rows:
        row["checkpoint_sha256"] = sha
        row["checkpoint_file"] = ckpt_final.name

    return pd.DataFrame(rows), pd.DataFrame(train_log)


## Étape 9 — Définir les configurations Vague 3


In [ ]:

configs = []
if RUN_UNETPP:
    configs.append({"model": "unetpp", "use_clahe": False})
if RUN_CONNECTED_UNETS:
    configs.append({"model": "connected_unets", "use_clahe": False})
if RUN_RESIDUAL_UNET:
    configs.append({"model": "residual_unet", "use_clahe": False})
if RUN_CONNECTED_CLAHE_ARM:
    configs.append({"model": "connected_unets_clahe", "use_clahe": True})

bad_tokens = ["clahe_aug", "pretrained", "swa"]
for c in configs:
    if any(tok in c["model"] for tok in bad_tokens):
        raise RuntimeError(f"Wrong config for Wave 3: {c}")

print("Wave 3 configs:")
for c in configs:
    print(c)

with open(OUT_DIR / "logs" / "wave3_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "configs": configs,
        "seeds": SEEDS,
        "run_training": RUN_TRAINING,
        "quick_debug": QUICK_DEBUG,
        "baseline_reference": BASELINE,
        "protocol": "threshold selected on CBIS validation only, then frozen for CBIS test and INbreast",
    }, f, indent=2, ensure_ascii=False)


## Étape 10 — Lancement entraînement + évaluation


In [ ]:

all_result_rows = []
all_train_logs = []

if RUN_TRAINING:
    for cfg in configs:
        for seed in SEEDS:
            result_df, log_df = train_one_run(cfg["model"], seed, use_clahe=cfg["use_clahe"])
            all_result_rows.append(result_df)
            all_train_logs.append(log_df)

    results_df = pd.concat(all_result_rows, ignore_index=True)
    trainlog_df = pd.concat(all_train_logs, ignore_index=True)
    results_df.to_csv(OUT_DIR / "wave3_results.csv", index=False)
    results_df.to_csv(OUT_DIR / "metrics" / "all_wave3_seed_summaries.csv", index=False)
    trainlog_df.to_csv(OUT_DIR / "training_log.csv", index=False)
    print("Saved wave3_results.csv and training_log.csv")
else:
    print("RUN_TRAINING=False: skipping long training. Run precheck/visualization only.")


## Étape 11 — Agrégation moyenne ± écart-type et comparaison à la baseline


In [ ]:

def format_mean_std(mean, std, digits=4):
    if pd.isna(mean): return "NA"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"

if RUN_TRAINING:
    results_df = pd.read_csv(OUT_DIR / "wave3_results.csv")
    metrics_cols = ["dice", "iou", "precision", "recall", "hd95", "asd", "empty_pred_rate", "selected_threshold", "n_params"]
    agg = results_df.groupby(["model", "split"], as_index=False).agg({c: ["mean", "std"] for c in metrics_cols})
    agg.columns = ["_".join([x for x in col if x]) for col in agg.columns.values]
    agg = agg.rename(columns={"model_": "model", "split_": "split"})
    agg.to_csv(OUT_DIR / "wave3_results_wide.csv", index=False)
    display(agg)

    rows = []
    for model in sorted(results_df["model"].unique()):
        sub_test = agg[(agg["model"] == model) & (agg["split"] == "test")]
        sub_ext = agg[(agg["model"] == model) & (agg["split"] == "external_inbreast")]
        sub_val = agg[(agg["model"] == model) & (agg["split"] == "validation")]
        if len(sub_test) and len(sub_ext):
            rtest = sub_test.iloc[0]
            rext = sub_ext.iloc[0]
            rval = sub_val.iloc[0] if len(sub_val) else None
            rows.append({
                "model": model,
                "val_dice": rval["dice_mean"] if rval is not None else np.nan,
                "cbis_test_dice": rtest["dice_mean"],
                "cbis_test_iou": rtest["iou_mean"],
                "cbis_test_hd95": rtest["hd95_mean"],
                "inbreast_dice": rext["dice_mean"],
                "inbreast_iou": rext["iou_mean"],
                "inbreast_hd95": rext["hd95_mean"],
                "delta_cbis_test_dice_vs_swin": rtest["dice_mean"] - BASELINE["cbis_test_dice"],
                "delta_inbreast_dice_vs_swin": rext["dice_mean"] - BASELINE["inbreast_dice"],
                "delta_inbreast_hd95_vs_swin": BASELINE["inbreast_hd95"] - rext["hd95_mean"],
                "selected_threshold_mean": rtest["selected_threshold_mean"],
                "n_params_mean": rtest["n_params_mean"],
            })
    compare_df = pd.DataFrame(rows).sort_values("val_dice", ascending=False)
    compare_df.to_csv(OUT_DIR / "wave3_comparison_vs_baseline.csv", index=False)
    display(compare_df)

    # Recommendation based on validation only, with final criterion checked on sealed sets
    best_by_val = compare_df.iloc[0].to_dict() if len(compare_df) else {}
    retained = []
    for _, row in compare_df.iterrows():
        if row["delta_inbreast_dice_vs_swin"] > 0 and row["delta_cbis_test_dice_vs_swin"] >= -0.001:
            retained.append(row["model"])

    md_lines = []
    md_lines.append("# Wave 3 architecture benchmark results\n")
    md_lines.append("## Protocol\n")
    md_lines.append("- Architectures tested under the same ROI/oracle-crop protocol as Track A.\n")
    md_lines.append("- Threshold selected only on CBIS-DDSM validation, then frozen for CBIS-DDSM test and INbreast.\n")
    md_lines.append("- No CLAHE and no strong augmentation in the main Wave 3 configurations.\n")
    md_lines.append("- Baseline retained for comparison: Swin-Tiny-U-Net scratch, CBIS test Dice 0.888 and INbreast Dice 0.849.\n")

    md_lines.append("\n## Aggregated results versus baseline\n")
    md_lines.append(compare_df.to_markdown(index=False, floatfmt=".4f"))

    md_lines.append("\n## Validation-selected ranking\n")
    if best_by_val:
        md_lines.append(f"- Best architecture by CBIS validation Dice: `{best_by_val['model']}`.\n")
        md_lines.append(f"- Its CBIS test Dice is {best_by_val['cbis_test_dice']:.4f}, delta vs baseline {best_by_val['delta_cbis_test_dice_vs_swin']:+.4f}.\n")
        md_lines.append(f"- Its INbreast Dice is {best_by_val['inbreast_dice']:.4f}, delta vs baseline {best_by_val['delta_inbreast_dice_vs_swin']:+.4f}.\n")

    md_lines.append("\n## Retention decision\n")
    if retained:
        md_lines.append("The following architecture(s) improve INbreast without degrading CBIS test and can be considered as controlled improvements:\n")
        for m in retained:
            md_lines.append(f"- `{m}`\n")
    else:
        md_lines.append("No Wave 3 architecture satisfies the criterion of improving INbreast without degrading CBIS test. The Swin-Tiny-U-Net scratch baseline remains the final Track A model.\n")

    md_lines.append("\n## Note on Connected-SegNets literature comparison\n")
    md_lines.append("Connected-SegNets/Connected-UNets are evaluated here under our sealed external protocol. Published Connected-SegNets scores are not directly comparable when their INbreast evaluation is not a fully sealed external holdout with threshold frozen from CBIS validation.\n")

    report_path = OUT_DIR / "wave3_results.md"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))
    print("Saved:", report_path)
    print("\n".join(md_lines[:20]))
else:
    print("RUN_TRAINING=False: no result aggregation.")


## Étape 12 — Figure de comparaison et ZIP léger


In [ ]:

if RUN_TRAINING:
    compare_df = pd.read_csv(OUT_DIR / "wave3_comparison_vs_baseline.csv")
    fig, ax = plt.subplots(figsize=(9, 5))
    labels = ["Swin scratch baseline"] + compare_df["model"].tolist()
    inbreast = [BASELINE["inbreast_dice"]] + compare_df["inbreast_dice"].tolist()
    cbis_test = [BASELINE["cbis_test_dice"]] + compare_df["cbis_test_dice"].tolist()
    x = np.arange(len(labels))
    width = 0.35
    ax.bar(x - width/2, cbis_test, width, label="CBIS test Dice")
    ax.bar(x + width/2, inbreast, width, label="INbreast Dice")
    ax.axhline(BASELINE["inbreast_dice"], linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0.70, 0.95)
    ax.set_ylabel("Dice")
    ax.set_title("Wave 3 architectures vs Swin-Tiny scratch baseline")
    ax.legend()
    plt.tight_layout()
    fig_path = OUT_DIR / "figures" / "02_wave3_architectures_vs_baseline.png"
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

    zip_path = Path("/kaggle/working/ROI256_WAVE3_ARCHITECTURES_RESULTS_LIGHT.zip")
    if zip_path.exists():
        zip_path.unlink()
    exclude_dirs = {"checkpoints", "probabilities"}
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for p in OUT_DIR.rglob("*"):
            if p.is_file():
                rel = p.relative_to(OUT_DIR)
                if any(part in exclude_dirs for part in rel.parts):
                    continue
                zf.write(p, arcname=str(Path(OUT_DIR.name) / rel))
    print("Saved light ZIP:", zip_path)
else:
    zip_path = Path("/kaggle/working/ROI256_WAVE3_ARCHITECTURES_PRECHECK_LIGHT.zip")
    if zip_path.exists():
        zip_path.unlink()
    exclude_dirs = {"checkpoints", "probabilities"}
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for p in OUT_DIR.rglob("*"):
            if p.is_file():
                rel = p.relative_to(OUT_DIR)
                if any(part in exclude_dirs for part in rel.parts):
                    continue
                zf.write(p, arcname=str(Path(OUT_DIR.name) / rel))
    print("Saved precheck light ZIP:", zip_path)
